# Ollama Text Assistant + RAG (single project)

One notebook that runs entirely on your machine with [Ollama](https://ollama.com):

1. Text generation, 2. General question answering, 3. Summarization, 4. Translation, 5. **Ask your document (RAG)**

**Setup (once):**
```
pip install ollama numpy
ollama pull llama3.2:1b
ollama pull hf.co/CompendiumLabs/bge-base-en-v1.5-gguf
```
Put `AI_desc.txt` (or any `.txt` file, set below) in the same folder as this notebook.

In [1]:
import re
import numpy as np
import ollama

LANGUAGE_MODEL = 'llama3.2:1b'
EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
DOCUMENT_FILE = 'AI_desc.txt'   # the knowledge base for option 5
TOP_K = 3                       # passages retrieved per question
MIN_SIMILARITY = 0.5            # ignore passages scoring below this (tune 0.3 - 0.7)

## Part 1: Text tasks

In [2]:
def generate_text(prompt, temperature=0.7):
    res = ollama.generate(model=LANGUAGE_MODEL, prompt=prompt,
                          options={'temperature': temperature})
    return res['response']


def answer_question(question):
    return generate_text(f'Answer the following question clearly and concisely:\n\n{question}')


def summarize_text(text):
    return generate_text(f'Summarize the following text in one sentence:\n\n{text}')


def translate_text(text, language):
    return generate_text(
        f'Translate the following text into {language}. Reply with the translation only.\n\n{text}')

## Part 2: RAG (retrieval-augmented generation)

1. **Chunk** the document into paragraphs
2. **Embed** every chunk once and keep the normalised vectors
3. **Retrieve** the chunks most similar to the question (cosine similarity)
4. **Generate** an answer from only those chunks

In [3]:
def load_chunks(path):
    with open(path, encoding='utf-8') as f:
        text = f.read()
    paragraphs = [re.sub(r'\s+', ' ', p).strip() for p in re.split(r'\n\s*\n', text)]
    return [p for p in paragraphs if p]          # skips blank lines


def embed(texts):
    vectors = np.array(ollama.embed(model=EMBEDDING_MODEL, input=texts)['embeddings'],
                       dtype=np.float32)
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)   # unit length


def retrieve(query, chunks, vectors, top_k=TOP_K):
    scores = vectors @ embed([query])[0]          # cosine similarity (vectors are unit length)
    best = np.argsort(-scores)[:top_k]
    return [(chunks[i], float(scores[i])) for i in best]

In [4]:
RAG_SYSTEM_PROMPT = (
    'You are a helpful assistant that answers questions using ONLY the context provided. '
    'If the context does not contain the answer, reply exactly: '
    '"I couldn\'t find that in the provided documents." '
    'Do not use outside knowledge and do not invent facts. Keep the answer concise.'
)


def rag_answer(question, chunks, vectors):
    hits = [h for h in retrieve(question, chunks, vectors) if h[1] >= MIN_SIMILARITY]
    if not hits:
        return 'No passage in the document was similar enough to your question.', []
    context = '\n\n'.join(f'[{i}] {text}' for i, (text, _) in enumerate(hits, 1))
    res = ollama.chat(model=LANGUAGE_MODEL, messages=[
        {'role': 'system', 'content': RAG_SYSTEM_PROMPT},
        {'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {question}'},
    ])
    return res['message']['content'], hits

In [5]:
try:
    chunks = load_chunks(DOCUMENT_FILE)
    vectors = embed(chunks)
    print(f'Indexed {len(chunks)} passages from {DOCUMENT_FILE}')
except FileNotFoundError:
    chunks, vectors = [], None
    print(f'{DOCUMENT_FILE} not found. Options 1-4 still work; add the file and re-run this cell for option 5.')

Indexed 7 passages from AI_desc.txt


## Part 3: Menu
Run this cell, pick an option, and it keeps asking until you enter `0`.

In [6]:
MENU = '''
OLLAMA TEXT ASSISTANT
==================================================
1. Text Generation
2. Answer Your Question
3. Summarize The Text
4. Translate The Text
5. Ask Your Document (RAG)
0. Quit
'''

while True:
    print(MENU)
    choice = input('enter your choice: ').strip()
    if choice == '0':
        print('Bye!')
        break
    sources = []
    try:
        if choice == '1':
            result = generate_text(input('enter the prompt: '))
        elif choice == '2':
            result = answer_question(input('state your question: '))
        elif choice == '3':
            result = summarize_text(input('enter the text to summarize: '))
        elif choice == '4':
            text = input('enter the text: ')
            result = translate_text(text, input('enter the language: '))
        elif choice == '5':
            if not chunks:
                result = f'No document loaded. Add {DOCUMENT_FILE} and re-run the indexing cell.'
            else:
                result, sources = rag_answer(input('ask about the document: '), chunks, vectors)
        else:
            result = 'wrong input'
    except Exception as e:
        result = f'Error: {e}\nIs Ollama running and are the models pulled?'

    print('-' * 50, 'RESULT', '-' * 50)
    print(result)
    if sources:
        print('\nSources:')
        for i, (text, score) in enumerate(sources, 1):
            print(f'  [{i}] similarity {score:.2f}: {text[:90]}...')


OLLAMA TEXT ASSISTANT
1. Text Generation
2. Answer Your Question
3. Summarize The Text
4. Translate The Text
5. Ask Your Document (RAG)
0. Quit

-------------------------------------------------- RESULT --------------------------------------------------
Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making. AI systems use algorithms and data to analyze and generate information, and can be used in a variety of applications, including:

* Virtual assistants (e.g. Siri, Alexa)
* Image recognition (e.g. facial recognition, self-driving cars)
* Language translation (e.g. Google Translate)
* Predictive analytics (e.g. stock market predictions)
* Chatbots (e.g. customer service)

AI systems can be categorized into two main types:

* Narrow AI: designed to perform a specific task, such as image recognition or language translation.
* General AI: capable 